
# Random Forest with Uncertainty for Catalytic Biomass Pyrolysis  
**Targets:** `Y`, `SB`, `ST`, `SX` (confirmed) — **WHSV is treated as an input feature**.  
**Split:** 85:15 Train:Test (shared across targets) • **CV:** (optional) not required for parity plots  
**Uncertainty:** 90% Prediction Interval (tree-quantile across RF trees)  
**Outputs:** Metrics table + per-target parity plots with a **combined** 90% PI band across train+test.

> This notebook is designed for publication. You can **tweak tick spacing and axis limits** in the configuration cell.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
import math

In [2]:
# --- Configuration ---
DATA_PATH = "curated_data_no_outlier.xlsx"
OUT_DIR = "rf_cv_with_plots"
PLOTS_DIR = f"{OUT_DIR}/plots"
Q_LOW, Q_HIGH = 5, 95 # 90% PI
RANDOM_STATE = 42
K_FOLDS = 5

# Plot tick controls (optional)
TICK_STEP_X = 20
TICK_STEP_Y = 20

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)


In [5]:
# --- Load Data ---
df = pd.read_excel(DATA_PATH)
targets = ["Y", "SB", "ST", "SX"]
extra_drop = ["Zeolite"] 
X_df = df.drop(columns=targets + extra_drop).select_dtypes(include=[np.number])
y_df = df[targets]

X_df

,H/C,Promoter,Ec,X,Si/Al,Acidity,SBET,Type,T,CB,WSHV
0,0.015840,0.00,0.00,0.00,30.0,0.4900,370.3,1,600,19.0,10.423077
1,0.015840,0.00,0.00,0.00,30.0,0.4900,370.3,1,500,19.0,10.423077
2,0.015840,0.00,0.00,0.00,30.0,0.4900,370.3,1,670,19.0,10.423077
3,-0.410623,0.00,0.00,0.00,24.0,0.8700,332.1,1,500,1.0,12.000000
4,-0.410623,0.25,3.49,1.90,24.0,0.7300,322.7,1,500,1.0,12.000000
...,...,...,...,...,...,...,...,...,...,...,...
238,0.407018,1.00,4.28,1.83,25.0,1.3000,380.2,0,550,1.0,8.000000
239,0.407018,4.00,1.35,1.65,25.0,0.8820,302.8,0,550,1.0,8.000000
240,0.407018,4.00,6.82,2.16,25.0,0.9592,292.4,0,550,1.0,8.000000
241,0.407018,4.00,2.81,1.81,25.0,1.3980,320.5,0,550,1.0,8.000000


In [6]:
# --- Train/Test Split ---
indices = np.arange(len(X_df))
train_idx, test_idx = train_test_split(indices, test_size=0.15, random_state=RANDOM_STATE)
X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]

In [7]:
# --- Utility Functions ---
def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


def tree_quantile_interval(rf_pipe, X, q_low=5, q_high=95):
    Xt = rf_pipe.named_steps["scaler"].transform(rf_pipe.named_steps["imputer"].transform(X))
    preds = np.vstack([est.predict(Xt) for est in rf_pipe.named_steps["rf"].estimators_])
    return np.percentile(preds, q_low, axis=0), np.percentile(preds, q_high, axis=0)


def set_ticks_and_limits(ax, tick_step_x=None, tick_step_y=None, x_lim=None, y_lim=None):
    if x_lim is not None:
        ax.set_xlim(*x_lim)
    if y_lim is not None:
        ax.set_ylim(*y_lim)
    if tick_step_x is not None:
        ax.xaxis.set_major_locator(MultipleLocator(tick_step_x))
    if tick_step_y is not None:
        ax.yaxis.set_major_locator(MultipleLocator(tick_step_y))


def parity_plot_with_combined_PI(
    target_name,
    y_train, y_test, yhat_train, yhat_test,
    q05_train, q95_train, q05_test, q95_test,
    out_path,
    tick_step_x=None, tick_step_y=None,
    x_lim=None, y_lim=None
):
    x_all = np.concatenate([y_train, y_test])
    lo_all = np.concatenate([q05_train, q05_test])
    hi_all = np.concatenate([q95_train, q95_test])
    order = np.argsort(x_all)
    x_sorted, lo_sorted, hi_sorted = x_all[order], lo_all[order], hi_all[order]

    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
    ax.fill_between(x_sorted, lo_sorted, hi_sorted, alpha=0.2, color="gray", label="90% PI")
    ax.scatter(y_train, yhat_train, label="Train", alpha=0.75, color="blue",  s=150)
    ax.scatter(y_test, yhat_test, label="Test", alpha=0.9, color="orange",  s=150)
    ax.plot([x_all.min(), x_all.max()], [x_all.min(), x_all.max()], 'k--')
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Parity — {target_name}")
    ax.legend()
    ax.set_aspect('equal')

    # ✅ Now uses passed-in limits, not global ones
    set_ticks_and_limits(ax, tick_step_x, tick_step_y, x_lim, y_lim)

    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)


In [8]:
# --- Main Loop with CV & Plotting ---
results = []

for target in targets:
    print(f"\nProcessing: {target}")
    y = y_df[target]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("rf", RandomForestRegressor(random_state=RANDOM_STATE))
    ])

    param_grid = {
        "rf__n_estimators": [100,200,300,400,500,600],
        "rf__min_samples_leaf": [1, 2, 4],
        "rf__max_depth":[10,20,30,40],
        "rf__min_samples_split":[2,4,6,8,]
        
    }

    grid = GridSearchCV(pipe, param_grid, cv=KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE),
                       scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    yhat_train = best_model.predict(X_train)
    yhat_test = best_model.predict(X_test)
    q05_tr, q95_tr = tree_quantile_interval(best_model, X_train, Q_LOW, Q_HIGH)
    q05_te, q95_te = tree_quantile_interval(best_model, X_test, Q_LOW, Q_HIGH)

# Per-target axis limits
    PLOT_LIMITS = {
        "Y":   {"x_lim": (0, 80), "y_lim": (0, 80)},
        "SB":  {"x_lim": (0, 60), "y_lim": (0, 60)},
        "ST":  {"x_lim": (0, 60), "y_lim": (0, 60)},
        "SX":  {"x_lim": (0, 60), "y_lim": (0, 60)},
}
    
    limits = PLOT_LIMITS.get(target, {})
    
    # Remove the x_lim and y_lim parameters as they're not accepted by the function
    # Either pass the limits differently or modify the function to accept these parameters
    parity_plot_with_combined_PI(
    target, y_train, y_test, yhat_train, yhat_test,
    q05_tr, q95_tr, q05_te, q95_te,
    f"{PLOTS_DIR}/parity_{target}_cv.svg",
    tick_step_x=TICK_STEP_X,
    tick_step_y=TICK_STEP_Y,
    x_lim=limits.get("x_lim"),
    y_lim=limits.get("y_lim")
)

    results.append({
        "Target": target,
        "Train_RMSE": rmse(y_train, yhat_train),
        "Train_R2": r2_score(y_train, yhat_train),
        "Test_RMSE": rmse(y_test, yhat_test),
        "Test_R2": r2_score(y_test, yhat_test),
        "Best_Params": grid.best_params_
    })

pd.DataFrame(results)


Processing: Y


C:\Users\msuva\anaconda3\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



Processing: SB

Processing: ST


C:\Users\msuva\anaconda3\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



Processing: SX


C:\Users\msuva\anaconda3\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


,Target,Train_RMSE,Train_R2,Test_RMSE,Test_R2,Best_Params
0,Y,3.507445,0.875078,4.220530,0.797135,"{'rf__max_depth': 10, 'rf__min_samples_leaf': ..."
1,SB,2.660094,0.887843,3.009566,0.869926,"{'rf__max_depth': 30, 'rf__min_samples_leaf': ..."
2,ST,3.016986,0.865963,3.887668,0.744975,"{'rf__max_depth': 20, 'rf__min_samples_leaf': ..."
3,SX,2.434868,0.923544,5.275250,0.781026,"{'rf__max_depth': 30, 'rf__min_samples_leaf': ..."



## Notes for Publication
- **Uncertainty metric**: 90% **Prediction Interval (PI)** from tree-quantile RF reflects variability of *new observations* (preferred in experimental settings).
- The **combined PI band** is computed across **both train and test** sets for a unified visual summary around the parity line.
- Use the configuration cell to adjust **tick spacing** (`TICK_STEP_X`, `TICK_STEP_Y`) and **axis limits** (`X_LIM`, `Y_LIM`) to meet figure size and journal style.
- Metrics included: RMSE and $R^2$ for **Train** and **Test** per target.


In [11]:
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# --- SHAP Directory ---
SHAP_DIR = f"{OUT_DIR}/shap_bars"
os.makedirs(SHAP_DIR, exist_ok=True)

def plot_shap_bar(shap_values, feature_names, target_name, save_path):
    # Compute mean absolute SHAP values
    shap_df = pd.DataFrame({
        "Feature": feature_names,
        "Mean |SHAP|": np.abs(shap_values).mean(axis=0)
    }).sort_values("Mean |SHAP|", ascending=True)

    plt.figure(figsize=(6, 6))
    plt.barh(shap_df["Feature"], shap_df["Mean |SHAP|"], color="skyblue")
    plt.xlabel("Mean |SHAP| value")
    plt.title(f"SHAP Feature Importance — {target_name}")
    plt.xlim(left=0)  # ensures bars start from zero
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved SHAP bar plot for {target_name} at:\n{save_path}")


# --- Run SHAP for each target ---
for model_result in results:
    target = model_result["Target"]
    best_params = model_result["Best_Params"]

    print(f"\nFitting model for SHAP (Target: {target}) with params: {best_params}")

    # Fit model on training data with best params
    model = RandomForestRegressor(
        n_estimators=best_params["rf__n_estimators"],
        min_samples_leaf=best_params["rf__min_samples_leaf"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Impute and scale manually to simplify SHAP
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    
    X_train_imp = imputer.fit_transform(X_df.iloc[train_idx])
    X_train_scl = scaler.fit_transform(X_train_imp)
    y_train = y_df[target].iloc[train_idx]
    
    model.fit(X_train_scl, y_train)

    # SHAP values
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train_scl)

    # Plot bar chart
    plot_shap_bar(shap_values, X_df.columns.tolist(), target, f"{SHAP_DIR}/shap_bar_{target}.svg")


Fitting model for SHAP (Target: Y) with params: {'rf__max_depth': 20, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 100}
Saved SHAP bar plot for Y at:
rf_cv_with_plots/shap_bars/shap_bar_Y.svg

Fitting model for SHAP (Target: SB) with params: {'rf__max_depth': 30, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 500}
Saved SHAP bar plot for SB at:
rf_cv_with_plots/shap_bars/shap_bar_SB.svg

Fitting model for SHAP (Target: ST) with params: {'rf__max_depth': 10, 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 2, 'rf__n_estimators': 100}
Saved SHAP bar plot for ST at:
rf_cv_with_plots/shap_bars/shap_bar_ST.svg

Fitting model for SHAP (Target: SX) with params: {'rf__max_depth': 30, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 300}
Saved SHAP bar plot for SX at:
rf_cv_with_plots/shap_bars/shap_bar_SX.svg
